## Warmup Exercise

You work for a delivery company that wants to optimize delivery routes to maximize efficiency. To help plan, you’ve been given a dataset showing the type and number of packages delivered to each address in a neighborhood.

This data has been read in below and is stored in the DataFrame `delivery_data`.

In [1]:
import pandas as pd
import re

In [2]:
delivery_data = pd.read_csv("../data/delivery_data.csv")
delivery_data = delivery_data.rename(columns={"Candy_Number": "Package_Number"})
delivery_data.head()

,Address,Package_Type,Package_Number
0,101 Pine St,Home Goods,1
1,102 Pine St,Clothing,3
2,123 Elm St,Books,3
3,124 Elm St,Electronics,2
4,125 Elm St,Clothing,1


## Part 1

To help with our planning, it would be useful to have not just the address but the street name. Create a "Street" column by extracting the street name from the address. For example, "101 Pine St" would be "Pine St" (or just "Pine").

In [3]:
# INITIALIZE LIST FOR STREET NAMES
street_list = []

# LOOP THROUGH DATAFRAME ROWS
for index, row in delivery_data.iterrows():
    # USE REGEX TO FIND ADDRESS PATTERN
    street = re.findall(pattern=r"\d+\s(.+)", string=row['Address'])
    street_list.append(street[0])

# ADD NEW COLUMN WITH LIST OF STREETS
delivery_data['Street'] = street_list
delivery_data.head()

,Address,Package_Type,Package_Number,Street
0,101 Pine St,Home Goods,1,Pine St
1,102 Pine St,Clothing,3,Pine St
2,123 Elm St,Books,3,Elm St
3,124 Elm St,Electronics,2,Elm St
4,125 Elm St,Clothing,1,Elm St


In [4]:
# ALTERNATE PANDAS BUILT-IN TO EXTRACT STREET
delivery_data['Address'].str.extract("\d+\s(.+)")

,0
0,Pine St
1,Pine St
2,Elm St
3,Elm St
4,Elm St
5,Birch St
6,Birch St
7,Birch St
8,Cedar St
9,Cedar St


## Part 2

You’ve been assigned to focus on Electronics deliveries.
Which street receives the most Electronics packages overall?
Remember that each house may receive a different number of packages, so make sure to include that in your calculation.

In [5]:
# CALCULATE THE STREET WITH MOST ELECTRONIC PACKAGES
electronics_df = (
    delivery_data
    [delivery_data['Package_Type'] == "Electronics"]
    .groupby("Street")
    ['Package_Number']
    .sum()
    .sort_values(ascending=False)
)
electronics_df.head(1)

Street
Cedar St    5
Name: Package_Number, dtype: int64

## Part 3

You’d also like to know which street offers the best delivery efficiency—that is, the most Electronics packages delivered per block traveled.

The distance (in blocks) from your distribution center to each street is stored in a separate DataFrame:

In [6]:
distances = pd.read_csv("../data/distances.csv")
distances.head()

,Street,Blocks_From_Center
0,Elm St,1
1,Maple St,1
2,Oak St,2
3,Pine St,2
4,Birch St,3


In [7]:
# MERGE ELECTRONICS DF WITH DISTANCES
delivery_distance_df = pd.merge(
    left=electronics_df,
    right=distances,
    how="inner",
    on="Street",
    validate="1:1"
)
delivery_distance_df.head()

,Street,Package_Number,Blocks_From_Center
0,Cedar St,5,3
1,Walnut St,4,4
2,Birch St,3,3
3,Elm St,2,1


In [8]:
efficiency = []
for index, row in delivery_distance_df.iterrows():
    efficiency.append(row['Package_Number'] / row['Blocks_From_Center'])

delivery_distance_df['Efficiency'] = efficiency
delivery_distance_df.sort_values(by="Efficiency", ascending=False).head(1)

,Street,Package_Number,Blocks_From_Center,Efficiency
3,Elm St,2,1,2.0
